# Sri Lankan House Price Prediction Using Ensemble Learning

## Introduction

This project investigates the application of ensemble machine learning techniques to predict residential property asking prices in Sri Lanka. The study compares Bagging, Boosting, Stacking, and fixed-weight hybrid ensemble approaches using a real-world Sri Lankan property advertisement dataset.

The implemented models include Random Forest, XGBoost, CatBoost, a Ridge Regression-based Stacking Ensemble, and two averaging-based hybrid ensembles. Model performance is evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R².

An interactive Streamlit dashboard has also been developed to visualize model performance and generate property-price predictions.

## Dataset

**Source:** Sri Lanka Property Ads Dataset — Kaggle  
**Dataset link:** https://www.kaggle.com/datasets/ivantha/sri-lanka-property-ads-dataset

The original dataset contains Sri Lankan property advertisements. Only houses listed for sale were selected for this study.

The initial filtering produced 51,890 advertisements. After handling invalid prices, investigating outliers, applying documented feature limits, and removing likely repeated advertisements, 33,337 records remained.

**Input features:** Location, number of bedrooms, number of bathrooms, house size in square feet, and land size in perches.

**Target variable:** Advertised house price in millions of Sri Lankan Rupees.

The dataset represents historical asking prices rather than verified transaction prices. Consequently, the predictions should not be interpreted as current market valuations.

In [ ]:
import pandas as pd

# Load only the first 1000 records for initial inspection
df = pd.read_csv('/content/properties.csv', nrows=1000)

# Display dataset dimensions
print("Dataset shape:", df.shape)

# Display column names
print("\nColumn names:")
print(df.columns.tolist())

# Display the first five records
print("\nFirst 5 records:")
display(df.head())


Dataset shape: (1000, 26)

Column names:
['ad_title', 'ad_description', 'details', 'slug', 'title', 'type', 'price', 'timestamp', 'posted_date', 'deactivation_date', 'category', 'parent_category', 'location', 'geo_region', 'area', 'is_delivery_free', 'is_doorstep_delivery', 'is_dsd_applicable', 'is_member', 'is_authorized_dealer', 'is_featured_member', 'is_verified', 'membership_level', 'member_since', 'properties', 'user']

First 5 records:


,ad_title,ad_description,details,slug,title,type,price,timestamp,posted_date,deactivation_date,...,is_doorstep_delivery,is_dsd_applicable,is_member,is_authorized_dealer,is_featured_member,is_verified,membership_level,member_since,properties,user
0,apartment for sale in colombo 05 | ikman,Project: Aquaria by Access Residencies.\n\nSq....,"Bedrooms: 3, Bathrooms: 2",apartment-for-sale-in-colombo-05-for-sale-colo...,apartment for sale in colombo 05,for_sale,"Rs 59,500,000",16 Feb 2:38 pm,2023-02-16T14:38:34+05:30,2022-01-31T04:43:32.000Z,...,False,False,True,False,False,True,plus,October 2016,"{'Address': 'kirimandala mawatha, colombo 05.'...",2a36021370cdd0f50da19f64d17ad43be687d48568ec12...
1,Land for Sale in Kurunegala | ikman.lk,à·à·à¶à·âà¶»à¶ºà·à¶±à· à¶¯à·à¶ºà·à¶«...,15.0 perches,land-for-sale-in-kurunegala-for-sale-kurunegal...,Land for Sale in Kurunegala,for_sale,"Rs 50,000 per perch",10 Nov 12:06 pm,2022-11-10T12:06:19+05:30,2022-07-30T09:30:41.000Z,...,False,False,True,False,False,False,premium,April 2017,"{'Address': 'kurunegala,malkaduwawa', 'Land ty...",f917ed789e936ada5530694ede4edde85fdb53f739c5f6...
2,Land for Sale in Kelaniya | ikman,Good Residential Bare Land in Kelaniya\nLocate...,18.5 perches,land-for-sale-in-kelaniya-for-sale-gampaha-766,Land for Sale in Kelaniya,for_sale,"Rs 1,600,000 per perch",30 Jan 8:54 am,2023-01-30T08:54:35+05:30,2022-07-30T09:31:26.000Z,...,False,False,True,False,False,True,premium,August 2016,"{'Address': 'Dalugama, Kelaniya', 'Land type':...",147c45b684bfeae3e3ffb0397b126d4e67392ad600e254...
3,Land for Sale in Kelaniya | ikman,Good Residential bare land in Kelaniya\nLocate...,12.0 perches,land-for-sale-in-kelaniya-for-sale-gampaha-765,Land for Sale in Kelaniya,for_sale,"Rs 1,700,000 per perch",30 Jan 8:54 am,2023-01-30T08:54:29+05:30,2022-07-30T09:24:20.000Z,...,False,False,True,False,False,True,premium,August 2016,"{'Address': 'Off Kohalwila Road, Dalugama - Ke...",147c45b684bfeae3e3ffb0397b126d4e67392ad600e254...
4,Semi Luxury House for Sale in Kadawatha | ikman,Semi Luxury House in Kadawatha\nLocated at Pin...,"Bedrooms: 5, Bathrooms: 4",semi-luxury-house-for-sale-in-kadawatha-for-sa...,Semi Luxury House for Sale in Kadawatha,for_sale,"Rs 55,000,000",30 Jan 8:54 am,2023-01-30T08:54:09+05:30,2021-10-03T08:33:36.000Z,...,False,False,True,False,False,True,premium,August 2016,"{'Address': 'Pinthaliya Road, Kadawatha', 'Bed...",147c45b684bfeae3e3ffb0397b126d4e67392ad600e254...


In [ ]:
print("1. Property categories:")
print(df['category'].value_counts(dropna=False))

print("\n2. Parent categories:")
print(df['parent_category'].value_counts(dropna=False))

print("\n3. Advertisement types:")
print(df['type'].value_counts(dropna=False))

print("\n4. Example property records:")
display(
    df[
        ['category', 'parent_category', 'type', 'price', 'details']
    ].head(15)
)

print("\n5. Example structured house properties:")
print(df.loc[4, 'properties'])


1. Property categories:
category
Houses For Sale                   347
Land For Sale                     262
Land                              220
Apartment Rentals                  50
Commercial Properties For Sale     39
Apartments For Sale                34
Commercial Property Rentals        21
House Rentals                      16
Holiday & Short-Term Rental        10
Room & Annex Rentals                1
Name: count, dtype: int64

2. Parent categories:
parent_category
Property    1000
Name: count, dtype: int64

3. Advertisement types:
type
for_sale    900
for_rent    100
Name: count, dtype: int64

4. Example property records:


,category,parent_category,type,price,details
0,Apartments For Sale,Property,for_sale,"Rs 59,500,000","Bedrooms: 3, Bathrooms: 2"
1,Land,Property,for_sale,"Rs 50,000 per perch",15.0 perches
2,Land For Sale,Property,for_sale,"Rs 1,600,000 per perch",18.5 perches
3,Land For Sale,Property,for_sale,"Rs 1,700,000 per perch",12.0 perches
4,Houses For Sale,Property,for_sale,"Rs 55,000,000","Bedrooms: 5, Bathrooms: 4"
5,Land,Property,for_sale,"Rs 2,600,000 total price",11.5 perches
6,Land,Property,for_sale,"Rs 4,700,000 total price",20.0 perches
7,Land,Property,for_sale,"Rs 16,000,000 total price",7.0 acres
8,Apartment Rentals,Property,for_rent,"Rs 700,000 /month","Beds: 3, Baths: 3"
9,Houses For Sale,Property,for_sale,"Rs 80,000,000","Bedrooms: 4, Bathrooms: 4"



5. Example structured house properties:
{'Address': 'Pinthaliya Road, Kadawatha', 'Bedrooms': '5', 'Bathrooms': '4', 'House size': '3,000.0 sqft', 'Land size': '27.5 perches'}


In [ ]:

# Filter houses from the full dataset

import pandas as pd

file_path = "/content/properties.csv"

# Read only the columns we need
selected_columns = [
    "category",
    "type",
    "price",
    "location",
    "geo_region",
    "details",
    "properties",
    "posted_date",
    "slug"
]

house_chunks = []

# Read the large CSV in smaller chunks
for chunk in pd.read_csv(
    file_path,
    usecols=selected_columns,
    dtype=str,
    chunksize=10000
):

    # Keep only houses for sale
    filtered = chunk[
        (chunk["category"] == "Houses For Sale") &
        (chunk["type"] == "for_sale")
    ]

    house_chunks.append(filtered)

# Combine all filtered records
houses = pd.concat(house_chunks, ignore_index=True)

print("Total houses for sale:", len(houses))
print("Number of columns:", houses.shape[1])

print("\nMissing prices:")
print(houses["price"].isna().sum())

print("\nSample house records:")
display(
    houses[
        ["price", "location", "geo_region", "details"]
    ].head(10)
)

print("\nExample property details:")
print(houses["properties"].dropna().iloc[0])

print("\nSample price formats:")
print(
    houses["price"]
    .dropna()
    .sample(
        n=min(10, houses["price"].notna().sum()),
        random_state=42
    )
    .tolist()
)


Total houses for sale: 51890
Number of columns: 9

Missing prices:
0

Sample house records:


,price,location,geo_region,details
0,"Rs 55,000,000",Kadawatha,LK-12,"Bedrooms: 5, Bathrooms: 4"
1,"Rs 80,000,000",Dehiwala,LK-11,"Bedrooms: 4, Bathrooms: 4"
2,"Rs 125,000,000",Mount Lavinia,LK-11,"Bedrooms: 4, Bathrooms: 2"
3,"Rs 58,000,000",Malabe,LK-11,"Bedrooms: 4, Bathrooms: 3"
4,"Rs 22,000,000",Kotte,LK-11,"Bedrooms: 4, Bathrooms: 3"
5,"Rs 58,000,000",Talawatugoda,LK-11,"Bedrooms: 4, Bathrooms: 3"
6,"Rs 95,000,000",Talawatugoda,LK-11,"Bedrooms: 4, Bathrooms: 3"
7,"Rs 65,000,000",Talawatugoda,LK-11,"Bedrooms: 5, Bathrooms: 4"
8,"Rs 95,000,000",Talawatugoda,LK-11,"Bedrooms: 4, Bathrooms: 3"
9,"Rs 75,000,000",Talawatugoda,LK-11,"Bedrooms: 4, Bathrooms: 3"



Example property details:
{'Address': 'Pinthaliya Road, Kadawatha', 'Bedrooms': '5', 'Bathrooms': '4', 'House size': '3,000.0 sqft', 'Land size': '27.5 perches'}

Sample price formats:
['Rs 13,800,000', 'Rs 8,500,000', 'Rs 60,000,000', 'Rs 34,000,000', 'Rs 52,000,000', 'Rs 7,000,000', 'Rs 48,000,000', 'Rs 225,000,000', 'Rs 19,500,000', 'Rs 165,000,000']


In [ ]:

# Extract numerical house features

import pandas as pd
import numpy as np
import ast
import re

# Safely convert the properties text into a dictionary
def parse_properties(value):
    try:
        result = ast.literal_eval(value)
        return result if isinstance(result, dict) else {}
    except (ValueError, SyntaxError, TypeError):
        return {}

# Convert an advertised price into LKR
def parse_price(value):
    text = str(value).strip()

    match = re.fullmatch(
        r"Rs\.?\s*([\d,]+(?:\.\d+)?)\s*(?:total price|negotiable)?",
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return np.nan

    return float(match.group(1).replace(",", ""))

# Extract a numerical value
def parse_number(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"\d[\d,]*(?:\.\d+)?", str(value))

    if not match:
        return np.nan

    return float(match.group().replace(",", ""))

# Convert property area to standard units
def parse_area(value, target_unit):
    if pd.isna(value):
        return np.nan

    text = str(value).lower().strip()
    number = parse_number(text)

    if pd.isna(number):
        return np.nan

    if "perch" in text:
        sqft = number * 272.25
    elif "acre" in text:
        sqft = number * 43560
    elif any(unit in text for unit in ["sqft", "sq ft", "square feet"]):
        sqft = number
    elif any(unit in text for unit in ["sqm", "sq m", "m²", "square meter"]):
        sqft = number * 10.7639
    else:
        return np.nan

    if target_unit == "perches":
        return sqft / 272.25

    return sqft

# Create a working copy
housing = houses.copy()

# Parse structured property details
parsed = housing["properties"].apply(parse_properties)

# Extract features
housing["price_lkr"] = housing["price"].apply(parse_price)

housing["bedrooms"] = parsed.apply(
    lambda x: parse_number(x.get("Bedrooms"))
)

housing["bathrooms"] = parsed.apply(
    lambda x: parse_number(x.get("Bathrooms"))
)

housing["house_size_sqft"] = parsed.apply(
    lambda x: parse_area(x.get("House size"), "sqft")
)

housing["land_size_perches"] = parsed.apply(
    lambda x: parse_area(x.get("Land size"), "perches")
)

# Inspect the results
columns = [
    "location",
    "price_lkr",
    "bedrooms",
    "bathrooms",
    "house_size_sqft",
    "land_size_perches"
]

print("Extracted house features:")
display(housing[columns].head(10))

print("\nMissing values after extraction:")
print(housing[columns].isna().sum())

print("\nUnrecognized price formats:")
print(
    housing.loc[housing["price_lkr"].isna(), "price"]
    .value_counts()
    .head(10)
)

print("\nDuplicate advertisement slugs:")
print(housing["slug"].duplicated().sum())

print("\nHouse price statistics (LKR):")
print(
    housing["price_lkr"].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    )
)


Extracted house features:


,location,price_lkr,bedrooms,bathrooms,house_size_sqft,land_size_perches
0,Kadawatha,55000000.0,5.0,4.0,3000.0,27.50
1,Dehiwala,80000000.0,4.0,4.0,3000.0,8.75
2,Mount Lavinia,125000000.0,4.0,2.0,5600.0,20.00
3,Malabe,58000000.0,4.0,3.0,4000.0,16.00
4,Kotte,22000000.0,4.0,3.0,2200.0,6.50
5,Talawatugoda,58000000.0,4.0,3.0,3000.0,16.00
6,Talawatugoda,95000000.0,4.0,3.0,3800.0,10.00
7,Talawatugoda,65000000.0,5.0,4.0,4778.0,7.50
8,Talawatugoda,95000000.0,4.0,3.0,3800.0,10.00
9,Talawatugoda,75000000.0,4.0,3.0,3500.0,10.00



Missing values after extraction:
location             0
price_lkr            2
bedrooms             0
bathrooms            0
house_size_sqft      0
land_size_perches    0
dtype: int64

Unrecognized price formats:
price
Negotiable    2
Name: count, dtype: int64

Duplicate advertisement slugs:
1

House price statistics (LKR):
count    5.188800e+04
mean     5.838041e+07
std      3.951135e+09
min      7.000000e+00
1%       3.143500e+06
5%       6.500000e+06
50%      2.850000e+07
95%      1.200000e+08
99%      2.250000e+08
max      9.000000e+11
Name: price_lkr, dtype: float64


In [ ]:

# Initial data cleaning

# Keep original housing DataFrame unchanged
cleaned = housing.copy()

print("Original records:", len(cleaned))

# Remove records without numerical prices
cleaned = cleaned.dropna(subset=["price_lkr"])

# Remove duplicate advertisements
cleaned = cleaned.drop_duplicates(
    subset=["slug"],
    keep="first"
)

print("Records after initial cleaning:", len(cleaned))

print("\nRemaining missing prices:")
print(cleaned["price_lkr"].isna().sum())

print("\nRemaining duplicate slugs:")
print(cleaned["slug"].duplicated().sum())


Original records: 51890
Records after initial cleaning: 51887

Remaining missing prices:
0

Remaining duplicate slugs:
0


In [ ]:

# Outlier investigation

print("Houses priced below Rs 1 million:")
print((cleaned["price_lkr"] < 1_000_000).sum())

print("\nHouses priced above Rs 1 billion:")
print((cleaned["price_lkr"] > 1_000_000_000).sum())

print("\n10 cheapest house advertisements:")
display(
    cleaned.nsmallest(10, "price_lkr")[
        ["location", "price", "price_lkr",
         "bedrooms", "house_size_sqft"]
    ]
)

print("\n10 most expensive house advertisements:")
display(
    cleaned.nlargest(10, "price_lkr")[
        ["location", "price", "price_lkr",
         "bedrooms", "house_size_sqft"]
    ]
)


Houses priced below Rs 1 million:
78

Houses priced above Rs 1 billion:
4

10 cheapest house advertisements:


,location,price,price_lkr,bedrooms,house_size_sqft
6203,Horana,Rs 7,7.0,3.0,1600.0
4544,Negombo,Rs 27,27.0,3.0,1600.0
48456,Katunayake,Rs 165,165.0,3.0,1400.0
28999,Piliyandala,Rs 325,325.0,4.0,2800.0
44583,Rajagiriya,Rs 370,370.0,5.0,3000.0
10056,Battaramulla,Rs 850,850.0,4.0,2200.0
9543,Panadura,"Rs 9,000",9000.0,4.0,1634.0
50076,Monaragala City,"Rs 12,000",12000.0,1.0,5000.0
14050,Divulapitiya,"Rs 12,345",12345.0,3.0,1250.0
18117,Anuradhapura City,"Rs 12,500",12500.0,3.0,800.0



10 most expensive house advertisements:


,location,price,price_lkr,bedrooms,house_size_sqft
39436,Dehiwala,"Rs 900,000,002,100",9.000000e+11,9.0,2100.0
2536,Talawatugoda,"Rs 2,100,000,005",2.100000e+09,5.0,7516.0
39271,Rajagiriya,"Rs 2,000,000,000",2.000000e+09,6.0,6155.0
24669,Nugegoda,"Rs 1,460,000,000",1.460000e+09,10.0,17500.0
11549,Colombo 7,"Rs 1,000,000,000",1.000000e+09,6.0,8000.0
34898,Colombo 7,"Rs 925,000,000",9.250000e+08,6.0,11715.0
38304,Colombo 7,"Rs 900,000,000",9.000000e+08,6.0,11715.0
15940,Colombo 7,"Rs 880,000,000",8.800000e+08,6.0,11000.0
3309,Seeduwa,"Rs 850,000,000",8.500000e+08,4.0,5048.0
11409,Colombo 7,"Rs 850,000,000",8.500000e+08,8.0,6500.0


In [ ]:

#  Filter suspicious house prices

print("Records before price filtering:", len(cleaned))

# Define the modeling price range
MIN_PRICE = 1_000_000
MAX_PRICE = 1_000_000_000

# Keep houses within the selected price range
housing_clean = cleaned[
    cleaned["price_lkr"].between(
        MIN_PRICE, MAX_PRICE
    )
].copy()

# Convert price to millions for easier interpretation
housing_clean["price_million"] = (
    housing_clean["price_lkr"] / 1_000_000
)

# Standardize location text
housing_clean["location"] = (
    housing_clean["location"]
    .astype("string")
    .str.strip()
)

print("Records after filtering:", len(housing_clean))

print(
    "Removed records:",
    len(cleaned) - len(housing_clean)
)

print("\nPrice statistics (LKR millions):")
print(housing_clean["price_million"].describe())

print("\nMissing feature values:")
features = [
    "location",
    "bedrooms",
    "bathrooms",
    "house_size_sqft",
    "land_size_perches"
]

print(housing_clean[features].isna().sum())

display(
    housing_clean[
        features + ["price_million"]
    ].head()
)


Records before price filtering: 51887
Records after filtering: 51805
Removed records: 82

Price statistics (LKR millions):
count    51805.000000
mean        40.993170
std         46.349717
min          1.000000
25%         17.000000
50%         28.500000
75%         47.500000
max       1000.000000
Name: price_million, dtype: float64

Missing feature values:
location             0
bedrooms             0
bathrooms            0
house_size_sqft      0
land_size_perches    0
dtype: int64


,location,bedrooms,bathrooms,house_size_sqft,land_size_perches,price_million
0,Kadawatha,5.0,4.0,3000.0,27.50,55.0
1,Dehiwala,4.0,4.0,3000.0,8.75,80.0
2,Mount Lavinia,4.0,2.0,5600.0,20.00,125.0
3,Malabe,4.0,3.0,4000.0,16.00,58.0
4,Kotte,4.0,3.0,2200.0,6.50,22.0


In [ ]:

# Inspect property feature ranges

numeric_features = [
    "bedrooms",
    "bathrooms",
    "house_size_sqft",
    "land_size_perches"
]

print("Numerical feature statistics:")
display(
    housing_clean[numeric_features].describe(
        percentiles=[0.01, 0.50, 0.95, 0.99]
    )
)

print("\nSuspicious feature counts:")

print(
    "Bedrooms > 20:",
    (housing_clean["bedrooms"] > 20).sum()
)

print(
    "Bathrooms > 20:",
    (housing_clean["bathrooms"] > 20).sum()
)

print(
    "House size <= 0:",
    (housing_clean["house_size_sqft"] <= 0).sum()
)

print(
    "House size > 30000 sqft:",
    (housing_clean["house_size_sqft"] > 30000).sum()
)

print(
    "Land size <= 0:",
    (housing_clean["land_size_perches"] <= 0).sum()
)

print(
    "Land size > 500 perches:",
    (housing_clean["land_size_perches"] > 500).sum()
)


Numerical feature statistics:


,bedrooms,bathrooms,house_size_sqft,land_size_perches
count,51805.000000,51805.000000,51805.000000,51805.000000
mean,3.785079,2.734041,2671.402373,19.382292
std,1.244780,1.340262,5615.769479,341.506176
min,1.000000,1.000000,0.000000,0.000000
1%,1.000000,1.000000,540.000000,2.500000
50%,4.000000,3.000000,2350.000000,10.000000
95%,6.000000,5.000000,5000.000000,29.000000
99%,8.000000,7.000000,8561.440000,85.000000
max,10.000000,10.000000,900000.000000,62500.000000



Suspicious feature counts:
Bedrooms > 20: 0
Bathrooms > 20: 0
House size <= 0: 21
House size > 30000 sqft: 56
Land size <= 0: 4
Land size > 500 perches: 90


In [ ]:

# Final feature cleaning

print("Records before feature filtering:", len(housing_clean))

# Define valid feature ranges
valid_records = (
    housing_clean["house_size_sqft"].between(300, 30000)
    &
    housing_clean["land_size_perches"].between(1, 500)
    &
    housing_clean["bedrooms"].between(1, 10)
    &
    housing_clean["bathrooms"].between(1, 10)
)

# Inspect excluded records
print("\nExample excluded records:")
display(
    housing_clean.loc[
        ~valid_records,
        [
            "location",
            "house_size_sqft",
            "land_size_perches",
            "price_million"
        ]
    ].head(10)
)

# Create final working dataset
housing_final = housing_clean.loc[valid_records].copy()

# Reset row indices
housing_final.reset_index(drop=True, inplace=True)

print("\nRecords after cleaning:", len(housing_final))

print(
    "Total records removed in this step:",
    len(housing_clean) - len(housing_final)
)

print("\nFinal dataset shape:")
print(housing_final.shape)

print("\nFinal feature statistics:")
display(
    housing_final[
        [
            "bedrooms",
            "bathrooms",
            "house_size_sqft",
            "land_size_perches",
            "price_million"
        ]
    ].describe()
)

print("\nPossible repeated property records:")
print(
    housing_final.duplicated(
        subset=[
            "location",
            "bedrooms",
            "bathrooms",
            "house_size_sqft",
            "land_size_perches",
            "price_lkr"
        ]
    ).sum()
)


Records before feature filtering: 51805

Example excluded records:


,location,house_size_sqft,land_size_perches,price_million
332,Nugegoda,70000.0,27.0,170.0
830,Panadura,2700.0,700.0,147.5
845,Athurugiriya,42000.0,10.0,41.0
1231,Horana,1000.0,1000.0,4.9
1276,Deniyaya,3000.0,800.0,32.5
1388,Malabe,3280.0,3280.0,46.0
2581,Rajagiriya,1800.0,3200.0,85.0
3342,Trincomalee City,1600.0,800.0,8.5
3497,Dehiwala,7.5,2900.0,98.0
3552,Piliyandala,8.2,1250.0,16.0



Records after cleaning: 51457
Total records removed in this step: 348

Final dataset shape:
(51457, 15)

Final feature statistics:


,bedrooms,bathrooms,house_size_sqft,land_size_perches,price_million
count,51457.000000,51457.000000,51457.000000,51457.000000,51457.000000
mean,3.787337,2.737820,2595.454327,12.869206,41.042622
std,1.242263,1.339358,1633.865817,16.555014,46.235057
min,1.000000,1.000000,300.000000,1.000000,1.000000
25%,3.000000,2.000000,1525.000000,7.300000,17.000000
50%,4.000000,3.000000,2370.000000,10.000000,28.500000
75%,4.000000,4.000000,3200.000000,12.500000,47.500000
max,10.000000,10.000000,30000.000000,480.000000,1000.000000



Possible repeated property records:
23266


In [ ]:

# Investigate repeated property records

duplicate_columns = [
    "location",
    "bedrooms",
    "bathrooms",
    "house_size_sqft",
    "land_size_perches",
    "price_lkr"
]

# Count occurrences of each feature-price combination
duplicate_groups = (
    housing_final
    .groupby(duplicate_columns, dropna=False)
    .size()
    .sort_values(ascending=False)
)

print("Total records:", len(housing_final))

print(
    "Unique feature-price combinations:",
    len(duplicate_groups)
)

print(
    "Repeated rows:",
    housing_final.duplicated(
        subset=duplicate_columns
    ).sum()
)

print("\n10 most frequently repeated combinations:")
display(duplicate_groups.head(10).reset_index(name="count"))

# Check whether full property details are repeated
print("\nRepeated location + properties + price:")
print(
    housing_final.duplicated(
        subset=["location", "properties", "price_lkr"]
    ).sum()
)

# Inspect advertisements from the most repeated group
top_group = duplicate_groups.index[0]

mask = pd.Series(True, index=housing_final.index)

for column, value in zip(duplicate_columns, top_group):
    mask &= housing_final[column].eq(value)

print("\nExamples from the most repeated group:")

display(
    housing_final.loc[
        mask,
        [
            "slug",
            "posted_date",
            "location",
            "price_lkr",
            "properties"
        ]
    ].head(10)
)


Total records: 51457
Unique feature-price combinations: 28191
Repeated rows: 23266

10 most frequently repeated combinations:


,location,bedrooms,bathrooms,house_size_sqft,land_size_perches,price_lkr,count
0,Negombo,3.0,2.0,1525.0,10.00,22500000.0,214
1,Talawatugoda,4.0,4.0,4500.0,10.00,75000000.0,178
2,Talawatugoda,4.0,4.0,3500.0,7.00,52000000.0,176
3,Nugegoda,6.0,5.0,3500.0,18.95,76000000.0,167
4,Negombo,3.0,2.0,3900.0,10.00,37000000.0,130
5,Talawatugoda,4.0,3.0,3500.0,8.00,48000000.0,117
6,Talawatugoda,5.0,5.0,7100.0,20.00,210000000.0,116
7,Talawatugoda,5.0,5.0,6000.0,19.00,210000000.0,116
8,Talawatugoda,4.0,4.0,3200.0,7.00,42500000.0,115
9,Negombo,3.0,2.0,1600.0,10.00,27000000.0,111



Repeated location + properties + price:
18120

Examples from the most repeated group:


,slug,posted_date,location,price_lkr,properties
3667,house-for-sale-negambo-for-sale-gampaha-718,2022-10-27T09:15:55+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
3687,design-house-for-sale-negambo-for-sale-gampaha,2022-10-27T20:30:55+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
3961,house-for-sale-negombo-for-sale-gampaha-10738,2022-10-28T15:26:02+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
4325,perfect-house-for-sale-negambo-for-sale-gampah...,2022-10-29T09:49:16+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
4355,house-for-sale-negambo-for-sale-gampaha-724,2022-10-29T09:09:35+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
4502,house-for-sale-negambo-for-sale-gampaha-728,2022-10-30T08:57:21+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
4554,a-house-for-sale-negambo-for-sale-gampaha-56,2022-10-30T01:05:56+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
4631,house-for-sale-negombo-for-sale-gampaha-10745,2022-10-30T16:22:11+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
4829,clear-deed-with-house-for-sale-in-negombo-for-...,2022-10-31T14:39:46+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."
4967,white-house-for-sale-negambo-for-sale-gampaha-56,2022-11-01T08:41:18+05:30,Negombo,22500000.0,"{'Address': 'Daluwakotuwa, Negambo', 'Bedrooms..."


In [ ]:

# Remove repeated advertisements

import pandas as pd

print("Records before deduplication:", len(housing_final))

# Create a copy
housing_unique = housing_final.copy()

# Convert advertisement date into datetime
housing_unique["posted_dt"] = pd.to_datetime(
    housing_unique["posted_date"],
    errors="coerce",
    utc=True
)

# Sort by posting date
housing_unique = housing_unique.sort_values(
    "posted_dt",
    na_position="first",
    kind="stable"
)

# Remove repeated property advertisements
housing_unique = housing_unique.drop_duplicates(
    subset=[
        "location",
        "properties",
        "price_lkr"
    ],
    keep="last"
)

# Remove temporary column
housing_unique = housing_unique.drop(
    columns=["posted_dt"]
)

# Reset index
housing_unique = housing_unique.reset_index(drop=True)

print("\nRecords after deduplication:", len(housing_unique))

print(
    "Removed repeated advertisements:",
    len(housing_final) - len(housing_unique)
)

print("\nRemaining repeated full property details + price:")
print(
    housing_unique.duplicated(
        subset=["location", "properties", "price_lkr"]
    ).sum()
)

# Inspect remaining identical feature-price combinations
model_columns = [
    "location",
    "bedrooms",
    "bathrooms",
    "house_size_sqft",
    "land_size_perches",
    "price_lkr"
]

print("\nRemaining repeated feature-price combinations:")
print(
    housing_unique.duplicated(
        subset=model_columns
    ).sum()
)

print("\nFinal dataset preview:")
display(
    housing_unique[
        [
            "location",
            "bedrooms",
            "bathrooms",
            "house_size_sqft",
            "land_size_perches",
            "price_million"
        ]
    ].head()
)


Records before deduplication: 51457

Records after deduplication: 33337
Removed repeated advertisements: 18120

Remaining repeated full property details + price:
0

Remaining repeated feature-price combinations:
5146

Final dataset preview:


,location,bedrooms,bathrooms,house_size_sqft,land_size_perches,price_million
0,Malabe,3.0,1.0,1300.0,12.5,15.5
1,Piliyandala,3.0,2.0,1532.0,8.5,21.5
2,Wennappuwa,3.0,1.0,2320.0,20.0,15.0
3,Ja-Ela,3.0,2.0,1425.0,9.0,17.5
4,Polgahawela,2.0,1.0,850.0,18.0,4.5


##Data Preprocessing and Cleaning

Data preprocessing was performed to improve data quality and prepare the dataset for machine learning.

### Feature Extraction

The original dataset contained property information in textual and dictionary-like formats. Relevant features were extracted and converted into numerical representations.

- **Price:** Converted from strings such as "Rs 55,000,000" into numerical values.
- **Bedrooms and Bathrooms:** Extracted as numerical features.
- **House Size:** Standardized to square feet.
- **Land Size:** Standardized to perches.
- **Location:** Retained as a categorical feature.

House prices were converted into millions of Sri Lankan Rupees to simplify interpretation.

###Data Cleaning and Outlier Handling

Only advertisements categorized as "Houses For Sale" were selected, resulting in 51,890 records.

Two records with non-numerical prices and one duplicate advertisement slug were removed, leaving 51,887 records.

A provisional price range of Rs 1 million to Rs 1 billion was applied to exclude 82 suspicious or extreme price records.

Additional feature filtering was performed using the following limits:

| Feature | Accepted Range |
|---|---|
| Bedrooms | 1–10 |
| Bathrooms | 1–10 |
| House Size | 300–30,000 sq. ft. |
| Land Size | 1–500 perches |

This filtering removed 348 records.

These limits were chosen to focus the experiment on more typical residential listings. They are not universal validity thresholds and may exclude some genuine properties.

###Duplicate Advertisement Handling

Repeated property advertisements were identified using location, complete structured property information, and advertised price.

This process removed 18,120 likely repeated advertisements.

The final dataset contained **33,337 records**.

Because identical property details do not necessarily identify the same physical house, the deduplication method has limitations.

###Dataset Splitting

The cleaned dataset was divided into training, validation, and testing subsets using `GroupShuffleSplit`.

| Dataset | Records | Purpose |
|---|---:|---|
| Training | 21,222 | Model training |
| Validation | 5,372 | Model comparison and selection |
| Testing | 6,743 | Final performance evaluation |

Group-aware splitting was applied to prevent identical input-feature combinations from appearing across the subsets.

This reduces potential data leakage, although it cannot guarantee that every repeated property advertisement has been identified.

###Feature Encoding

One-Hot Encoding was applied to the categorical location feature for Random Forest and XGBoost.

CatBoost was configured to handle location as a categorical feature directly.

Preprocessing was included in model pipelines to ensure consistent transformations and prevent validation or test data from influencing fitted preprocessing parameters.

In [ ]:

# Prepare training, validation and test sets

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Define input features
feature_columns = [
    "location",
    "bedrooms",
    "bathrooms",
    "house_size_sqft",
    "land_size_perches"
]

# Define input (X) and target (y)
X = housing_unique[feature_columns].copy()
y = housing_unique["price_million"].copy()

# Standardize location formatting
X["location"] = (
    X["location"].astype(str).str.strip()
)

# Group identical input-feature combinations
groups = X.groupby(
    feature_columns, dropna=False, sort=False
).ngroup()

# First split: 80% development, 20% testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

development_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_dev = X.iloc[development_idx]
y_dev = y.iloc[development_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

# Second split: training and validation
dev_groups = groups.iloc[development_idx]

val_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    val_splitter.split(X_dev, y_dev, dev_groups)
)

X_train = X_dev.iloc[train_idx]
y_train = y_dev.iloc[train_idx]

X_val = X_dev.iloc[val_idx]
y_val = y_dev.iloc[val_idx]

# Display results
print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))

print("\nTotal records:",
      len(X_train) + len(X_val) + len(X_test))

print("\nInput features:")
print(X_train.columns.tolist())

print("\nTarget variable:")
print(y_train.name)

# Verify no identical feature groups cross the splits
train_groups = set(groups.loc[X_train.index])
val_groups = set(groups.loc[X_val.index])
test_groups = set(groups.loc[X_test.index])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("\nGroup overlap check: PASSED")


Training records: 21222
Validation records: 5372
Testing records: 6743

Total records: 33337

Input features:
['location', 'bedrooms', 'bathrooms', 'house_size_sqft', 'land_size_perches']

Target variable:
price_million

Group overlap check: PASSED


In [ ]:

# Feature preprocessing

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Define categorical and numerical columns
categorical_features = ["location"]

numerical_features = [
    "bedrooms",
    "bathrooms",
    "house_size_sqft",
    "land_size_perches"
]

# Pipeline for numerical features
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Pipeline for categorical features
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully!")


Preprocessing pipeline created successfully!


## Ensemble Learning Methodology

This project evaluates three major ensemble learning approaches: Bagging, Boosting, and Stacking. Additional hybrid ensembles were implemented to investigate whether combining different models improves house-price prediction performance.

A simple baseline model was also developed to provide a reference for evaluating the ensemble techniques.

###  Baseline Model

A `DummyRegressor` was implemented using the median training-house price as a constant prediction.

This model provides a reference point for measuring the improvements achieved by more advanced machine learning algorithms.

### Random Forest — Bagging

Random Forest is an ensemble learning algorithm that combines predictions from multiple Decision Trees.

Each tree is trained using a bootstrap sample of the training data, while random feature selection introduces diversity among trees. For regression, predictions from individual trees are averaged.

The model was configured with the following hyperparameters:

| Hyperparameter | Value |
|---|---|
| Number of trees (`n_estimators`) | 100 |
| Maximum tree depth (`max_depth`) | 18 |
| Minimum samples per leaf (`min_samples_leaf`) | 3 |
| Random state | 42 |

Random Forest was selected to investigate the effectiveness of Bagging for predicting Sri Lankan house prices.

### XGBoost — Boosting

XGBoost is an optimized gradient-boosting algorithm that builds Decision Trees sequentially to improve predictions.

Unlike Bagging, where models are trained independently, Boosting progressively adds models to reduce the current ensemble's prediction errors.

The XGBoost model used the following configuration:

| Hyperparameter | Value |
|---|---|
| Number of trees | 200 |
| Maximum depth | 6 |
| Learning rate | 0.05 |
| Subsample ratio | 0.8 |
| Feature subsample ratio | 0.8 |
| Tree method | Histogram-based |

One-Hot Encoding was applied to the location feature through a preprocessing pipeline.

XGBoost was included to compare the predictive performance of Boosting against Random Forest's Bagging approach.

### CatBoost — Categorical Boosting

CatBoost is a gradient-boosting algorithm designed to handle categorical features effectively.

The model was trained using the original categorical location feature rather than applying external One-Hot Encoding.

The following configuration was used:

| Hyperparameter | Value |
|---|---|
| Training iterations | 350 |
| Tree depth | 6 |
| Learning rate | 0.05 |
| Loss function | RMSE |
| Categorical feature | Location |
| Random seed | 42 |

CatBoost was selected to investigate whether its categorical-feature handling could provide advantages over the other ensemble models.

### Stacking Ensemble

A Stacking Ensemble was developed by combining the predictions of Random Forest, XGBoost, and CatBoost.

The architecture consists of two levels:

**Level 1 — Base Learners**
- Random Forest Regressor
- XGBoost Regressor
- CatBoost Regressor

**Level 2 — Meta-Learner**
- Ridge Regression (`alpha = 10.0`)

Three-fold group-aware cross-validation was implemented using `GroupKFold` to generate out-of-fold predictions from the training dataset.

These predictions were used to train the Ridge Regression meta-learner.

Group-aware cross-validation reduced leakage caused by identical input-feature combinations appearing in both the training and held-out folds.

The trained meta-learner then combined predictions from the three base models to produce the final stacking prediction.

The objective was to determine whether learning a combination of different ensemble models could improve predictive performance compared with individual models.

###  Hybrid Ensemble Models

Two additional hybrid ensemble approaches were implemented.

**Hybrid 1 — Random Forest + XGBoost**

Predictions from Random Forest and XGBoost were combined using equal weights:

\[
\hat{y}_{hybrid} = 0.5\hat{y}_{RF} + 0.5\hat{y}_{XGB}
\]

**Hybrid 2 — Equal-Weight Ensemble**

Predictions from Random Forest, XGBoost, and CatBoost were averaged:

\[
\hat{y}_{equal} = \frac{\hat{y}_{RF}+\hat{y}_{XGB}+\hat{y}_{CAT}}{3}
\]

Unlike Stacking, these hybrid approaches use fixed combination weights without training an additional meta-learner.

Their performance was compared against the individual models and the Stacking Ensemble to investigate whether more complex model combinations provide meaningful improvements.

### Model Evaluation

All models were evaluated using three regression metrics:

- **Mean Absolute Error (MAE):** Measures the average absolute difference between actual and predicted house prices.
- **Root Mean Squared Error (RMSE):** Measures prediction error while assigning greater penalties to larger errors.
- **R² Score:** Measures predictive performance relative to a mean-based reference.

MAE and RMSE are expressed in millions of Sri Lankan Rupees.

The validation dataset was used for model comparison and selection, while the separate test dataset was reserved for final evaluation.

This approach supports a more reliable assessment of model generalization to unseen property records.

In [ ]:
# Train baseline and Random Forest

import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Baseline model
baseline_model = DummyRegressor(strategy="median")

baseline_model.fit(X_train, y_train)

# 2. Random Forest model with preprocessing
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=18,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training Random Forest...")

rf_model.fit(X_train, y_train)

print("Random Forest training completed!")

# 3. Evaluation function
def evaluate_model(name, model, X_data, y_data):
    predictions = model.predict(X_data)

    mae = mean_absolute_error(y_data, predictions)
    rmse = np.sqrt(mean_squared_error(y_data, predictions))
    r2 = r2_score(y_data, predictions)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

# 4. Evaluate on validation data
results = []

results.append(
    evaluate_model(
        "Baseline",
        baseline_model,
        X_val,
        y_val
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        rf_model,
        X_val,
        y_val
    )
)

# 5. Display results
comparison_df = pd.DataFrame(results)

print("\nValidation Performance Comparison:")
display(comparison_df.round(4))

# 6. Check potential overfitting
rf_train_predictions = rf_model.predict(X_train)

train_mae = mean_absolute_error(
    y_train, rf_train_predictions
)

val_mae = mean_absolute_error(
    y_val, rf_model.predict(X_val)
)

print("\nRandom Forest Training MAE:", round(train_mae, 4))
print("Random Forest Validation MAE:", round(val_mae, 4))


Training Random Forest...
Random Forest training completed!

Validation Performance Comparison:


,Model,MAE,RMSE,R2
0,Baseline,22.6346,46.9156,-0.0615
1,Random Forest,11.2937,30.7124,0.5451



Random Forest Training MAE: 8.1076
Random Forest Validation MAE: 11.2937


In [ ]:

# Train XGBoost Regressor

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Create XGBoost pipeline
xgb_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training XGBoost...")

# Train only on training data
xgb_model.fit(X_train, y_train)

print("XGBoost training completed!")

# Evaluate on validation data
xgb_results = evaluate_model(
    "XGBoost",
    xgb_model,
    X_val,
    y_val
)

# Compare all models
comparison_df = pd.DataFrame([
    evaluate_model(
        "Baseline", baseline_model, X_val, y_val
    ),
    evaluate_model(
        "Random Forest", rf_model, X_val, y_val
    ),
    xgb_results
])

print("\nModel Comparison:")
display(comparison_df.round(4))

# Check XGBoost overfitting
xgb_train_mae = mean_absolute_error(
    y_train,
    xgb_model.predict(X_train)
)

xgb_val_mae = mean_absolute_error(
    y_val,
    xgb_model.predict(X_val)
)

print("\nXGBoost Training MAE:",
      round(xgb_train_mae, 4))

print("XGBoost Validation MAE:",
      round(xgb_val_mae, 4))


Training XGBoost...
XGBoost training completed!

Model Comparison:


,Model,MAE,RMSE,R2
0,Baseline,22.6346,46.9156,-0.0615
1,Random Forest,11.2937,30.7124,0.5451
2,XGBoost,10.9345,27.4908,0.6355



XGBoost Training MAE: 9.4009
XGBoost Validation MAE: 10.9345


In [ ]:

!pip -q install catboost


In [ ]:

# Train CatBoost Regressor

import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# CatBoost supports categorical features directly
cat_model = CatBoostRegressor(
    iterations=350,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    cat_features=["location"],
    random_seed=42,
    verbose=False,
    thread_count=-1
)

print("Training CatBoost...")

# Train on the same training dataset
cat_model.fit(X_train, y_train)

print("CatBoost training completed!")

# Evaluate on validation data
cat_results = evaluate_model(
    "CatBoost",
    cat_model,
    X_val,
    y_val
)

# Compare all trained models
comparison_df = pd.DataFrame([
    evaluate_model("Baseline", baseline_model, X_val, y_val),
    evaluate_model("Random Forest", rf_model, X_val, y_val),
    evaluate_model("XGBoost", xgb_model, X_val, y_val),
    cat_results
])

print("\nValidation Model Comparison:")
display(comparison_df.round(4))

# Check training and validation errors
cat_train_mae = mean_absolute_error(
    y_train,
    cat_model.predict(X_train)
)

cat_val_mae = mean_absolute_error(
    y_val,
    cat_model.predict(X_val)
)

print("\nCatBoost Training MAE:",
      round(cat_train_mae, 4))

print("CatBoost Validation MAE:",
      round(cat_val_mae, 4))


Training CatBoost...
CatBoost training completed!

Validation Model Comparison:


,Model,MAE,RMSE,R2
0,Baseline,22.6346,46.9156,-0.0615
1,Random Forest,11.2937,30.7124,0.5451
2,XGBoost,10.9345,27.4908,0.6355
3,CatBoost,11.0137,28.3088,0.6135



CatBoost Training MAE: 10.4246
CatBoost Validation MAE: 11.0137


In [ ]:

# Random Forest + XGBoost + CatBoost + Ridge Meta-Learner

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from catboost import CatBoostRegressor


# ------------------------------------------------------------
# 1. Define our previously trained base models
# ------------------------------------------------------------

base_models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "CatBoost": cat_model
}


# ------------------------------------------------------------
# 2. Create groups using input features
# ------------------------------------------------------------

train_groups = X_train.groupby(
    feature_columns,
    dropna=False,
    sort=False
).ngroup()


# ------------------------------------------------------------
# 3. Prepare out-of-fold prediction storage
# ------------------------------------------------------------

oof_predictions = pd.DataFrame(
    index=X_train.index,
    columns=list(base_models.keys()),
    dtype=float
)


# ------------------------------------------------------------
# 4. Create group-aware cross-validation
# ------------------------------------------------------------

gkf = GroupKFold(n_splits=3)

print("Generating out-of-fold predictions...")


# ------------------------------------------------------------
# 5. Train models across the three folds
# ------------------------------------------------------------

for fold, (train_idx, holdout_idx) in enumerate(
    gkf.split(X_train, y_train, train_groups),
    start=1
):

    print(f"\nProcessing fold {fold}/3")

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]

    X_fold_holdout = X_train.iloc[holdout_idx]

    for name, model in base_models.items():

        # FIX: CatBoost cannot be cloned normally
        # in some versions of scikit-learn.

        if name == "CatBoost":

            fold_model = CatBoostRegressor(
                iterations=350,
                depth=6,
                learning_rate=0.05,
                loss_function="RMSE",
                cat_features=["location"],
                random_seed=42,
                verbose=False,
                thread_count=-1
            )

        else:

            # Clone Random Forest and XGBoost pipelines
            fold_model = clone(model)

        # Train a fresh model for this fold
        fold_model.fit(
            X_fold_train,
            y_fold_train
        )

        # Generate predictions for unseen fold
        predictions = fold_model.predict(
            X_fold_holdout
        )

        # Store out-of-fold predictions
        oof_predictions.iloc[
            holdout_idx,
            oof_predictions.columns.get_loc(name)
        ] = predictions

        print(f"  {name}: completed")


# ------------------------------------------------------------
# 6. Verify out-of-fold predictions
# ------------------------------------------------------------

assert not oof_predictions.isna().any().any(), (
    "Some out-of-fold predictions are missing!"
)

print("\nAll out-of-fold predictions generated successfully!")


# ------------------------------------------------------------
# 7. Train Ridge Regression meta-learner
# ------------------------------------------------------------

meta_model = Ridge(alpha=10.0)

meta_model.fit(
    oof_predictions,
    y_train
)

print("\nMeta-learner training completed!")


# ------------------------------------------------------------
# 8. Generate predictions using trained base models
# ------------------------------------------------------------

validation_predictions = pd.DataFrame(
    {
        name: model.predict(X_val)
        for name, model in base_models.items()
    },
    index=X_val.index
)


# ------------------------------------------------------------
# 9. Generate final stacking predictions
# ------------------------------------------------------------

stack_predictions = meta_model.predict(
    validation_predictions
)


# ------------------------------------------------------------
# 10. Evaluate stacking model
# ------------------------------------------------------------

stack_mae = mean_absolute_error(
    y_val,
    stack_predictions
)

stack_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        stack_predictions
    )
)

stack_r2 = r2_score(
    y_val,
    stack_predictions
)


# ------------------------------------------------------------
# 11. Display stacking results
# ------------------------------------------------------------

print("\n====================================")
print("STACKING ENSEMBLE VALIDATION RESULTS")
print("====================================")

print("MAE :", round(stack_mae, 4))
print("RMSE:", round(stack_rmse, 4))
print("R2  :", round(stack_r2, 4))


# ------------------------------------------------------------
# 12. Display meta-learner coefficients
# ------------------------------------------------------------

print("\nMeta-learner coefficients:")

coefficients = pd.Series(
    meta_model.coef_,
    index=list(base_models.keys())
)

display(coefficients.round(4))


# ------------------------------------------------------------
# 13. Compare stacking with previous models
# ------------------------------------------------------------

stacking_result = {
    "Model": "Stacking",
    "MAE": stack_mae,
    "RMSE": stack_rmse,
    "R2": stack_r2
}

comparison_with_stacking = pd.concat(
    [
        comparison_df[
            comparison_df["Model"] != "Stacking"
        ],
        pd.DataFrame([stacking_result])
    ],
    ignore_index=True
)

comparison_with_stacking = (
    comparison_with_stacking
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("\n====================================")
print("FINAL VALIDATION MODEL COMPARISON")
print("====================================")

display(comparison_with_stacking.round(4))

print(
    "\nBest model by validation MAE:",
    comparison_with_stacking.iloc[0]["Model"]
)


Generating out-of-fold predictions...

Processing fold 1/3
  Random Forest: completed
  XGBoost: completed
  CatBoost: completed

Processing fold 2/3
  Random Forest: completed
  XGBoost: completed
  CatBoost: completed

Processing fold 3/3
  Random Forest: completed
  XGBoost: completed
  CatBoost: completed

All out-of-fold predictions generated successfully!

Meta-learner training completed!

STACKING ENSEMBLE VALIDATION RESULTS
MAE : 10.6714
RMSE: 27.3217
R2  : 0.64

Meta-learner coefficients:


,0
Random Forest,0.1038
XGBoost,0.5623
CatBoost,0.3642



FINAL VALIDATION MODEL COMPARISON


,Model,MAE,RMSE,R2
0,Stacking,10.6714,27.3217,0.6400
1,XGBoost,10.9345,27.4908,0.6355
2,CatBoost,11.0137,28.3088,0.6135
3,Random Forest,11.2937,30.7124,0.5451
4,Baseline,22.6346,46.9156,-0.0615



Best model by validation MAE: Stacking


In [ ]:

# HYBRID ENSEMBLE EXPERIMENT


import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Generate predictions using trained models

rf_predictions = rf_model.predict(X_val)
xgb_predictions = xgb_model.predict(X_val)
cat_predictions = cat_model.predict(X_val)


# 2. Hybrid Model 1: Random Forest + XGBoost

hybrid_rf_xgb = (
    0.5 * rf_predictions +
    0.5 * xgb_predictions
)


# 3. Hybrid Model 2: Average of all three models

hybrid_average = (
    rf_predictions +
    xgb_predictions +
    cat_predictions
) / 3


# 4. Evaluation function for predictions

def evaluate_hybrid(name, predictions):
    return {
        "Model": name,
        "MAE": mean_absolute_error(
            y_val, predictions
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_val, predictions
            )
        ),
        "R2": r2_score(
            y_val, predictions
        )
    }


# 5. Evaluate both hybrids

hybrid_results = pd.DataFrame([
    evaluate_hybrid(
        "Hybrid RF + XGBoost",
        hybrid_rf_xgb
    ),
    evaluate_hybrid(
        "Equal-Weight Ensemble",
        hybrid_average
    )
])


# 6. Compare with previous models

complete_comparison = pd.concat(
    [
        comparison_with_stacking,
        hybrid_results
    ],
    ignore_index=True
)

complete_comparison = (
    complete_comparison
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("COMPLETE VALIDATION MODEL COMPARISON")
display(complete_comparison.round(4))

print(
    "\nBest model based on validation MAE:",
    complete_comparison.iloc[0]["Model"]
)


COMPLETE VALIDATION MODEL COMPARISON


,Model,MAE,RMSE,R2
0,Stacking,10.6714,27.3217,0.6400
1,Equal-Weight Ensemble,10.6773,27.8385,0.6263
2,Hybrid RF + XGBoost,10.9111,28.5140,0.6079
3,XGBoost,10.9345,27.4908,0.6355
4,CatBoost,11.0137,28.3088,0.6135
5,Random Forest,11.2937,30.7124,0.5451
6,Baseline,22.6346,46.9156,-0.0615



Best model based on validation MAE: Stacking


In [ ]:

# EVALUATION ON UNSEEN TEST DATA

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Evaluating models on the test dataset...")

# ------------------------------------------------------------
# 1. Generate predictions from individual models
# ------------------------------------------------------------

test_predictions = {
    "Baseline": baseline_model.predict(X_test),
    "Random Forest": rf_model.predict(X_test),
    "XGBoost": xgb_model.predict(X_test),
    "CatBoost": cat_model.predict(X_test)
}


# ------------------------------------------------------------
# 2. Generate hybrid predictions
# ------------------------------------------------------------

test_predictions["Hybrid RF + XGBoost"] = (
    0.5 * test_predictions["Random Forest"]
    + 0.5 * test_predictions["XGBoost"]
)

test_predictions["Equal-Weight Ensemble"] = (
    test_predictions["Random Forest"]
    + test_predictions["XGBoost"]
    + test_predictions["CatBoost"]
) / 3


# ------------------------------------------------------------
# 3. Generate Stacking predictions
# ------------------------------------------------------------

# Use the exact same base-model order
# used to train the Ridge meta-learner.

stacking_test_inputs = pd.DataFrame(
    {
        name: test_predictions[name]
        for name in base_models.keys()
    },
    index=X_test.index
)

test_predictions["Stacking"] = meta_model.predict(
    stacking_test_inputs
)


# ------------------------------------------------------------
# 4. Evaluate all model predictions
# ------------------------------------------------------------

test_results = []

for name, predictions in test_predictions.items():

    mae = mean_absolute_error(
        y_test, predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test, predictions
        )
    )

    r2 = r2_score(
        y_test, predictions
    )

    test_results.append({
        "Model": name,
        "Test_MAE": mae,
        "Test_RMSE": rmse,
        "Test_R2": r2
    })


# ------------------------------------------------------------
# 5. Create test comparison table
# ------------------------------------------------------------

test_comparison = pd.DataFrame(test_results)

print("\nFINAL TEST PERFORMANCE:")
display(
    test_comparison
    .sort_values("Test_MAE")
    .round(4)
)


# ------------------------------------------------------------
# 6. Compare validation and testing results
# ------------------------------------------------------------

validation_results = complete_comparison.rename(
    columns={
        "MAE": "Validation_MAE",
        "RMSE": "Validation_RMSE",
        "R2": "Validation_R2"
    }
)

final_comparison = validation_results.merge(
    test_comparison,
    on="Model"
)

# Preserve validation-based ranking
final_comparison = final_comparison.sort_values(
    "Validation_MAE"
).reset_index(drop=True)

print("\nVALIDATION VS TEST COMPARISON:")
display(final_comparison.round(4))


# ------------------------------------------------------------
# 7. Report validation-selected model
# ------------------------------------------------------------

selected_model = final_comparison.iloc[0]

print("\n====================================")
print("VALIDATION-SELECTED MODEL")
print("====================================")

print("Model:", selected_model["Model"])

print(
    "Validation MAE:",
    round(selected_model["Validation_MAE"], 4)
)

print(
    "Test MAE:",
    round(selected_model["Test_MAE"], 4)
)

print(
    "Test RMSE:",
    round(selected_model["Test_RMSE"], 4)
)

print(
    "Test R2:",
    round(selected_model["Test_R2"], 4)
)


Evaluating models on the test dataset...

FINAL TEST PERFORMANCE:


,Model,Test_MAE,Test_RMSE,Test_R2
5,Equal-Weight Ensemble,9.9123,23.2264,0.7117
6,Stacking,9.9309,22.9200,0.7192
4,Hybrid RF + XGBoost,10.1125,23.5192,0.7044
2,XGBoost,10.2573,23.1249,0.7142
3,CatBoost,10.3163,24.5849,0.6770
1,Random Forest,10.3648,25.0012,0.6659
0,Baseline,21.7358,44.4029,-0.0538



VALIDATION VS TEST COMPARISON:


,Model,Validation_MAE,Validation_RMSE,Validation_R2,Test_MAE,Test_RMSE,Test_R2
0,Stacking,10.6714,27.3217,0.6400,9.9309,22.9200,0.7192
1,Equal-Weight Ensemble,10.6773,27.8385,0.6263,9.9123,23.2264,0.7117
2,Hybrid RF + XGBoost,10.9111,28.5140,0.6079,10.1125,23.5192,0.7044
3,XGBoost,10.9345,27.4908,0.6355,10.2573,23.1249,0.7142
4,CatBoost,11.0137,28.3088,0.6135,10.3163,24.5849,0.6770
5,Random Forest,11.2937,30.7124,0.5451,10.3648,25.0012,0.6659
6,Baseline,22.6346,46.9156,-0.0615,21.7358,44.4029,-0.0538



VALIDATION-SELECTED MODEL
Model: Stacking
Validation MAE: 10.6714
Test MAE: 9.9309
Test RMSE: 22.92
Test R2: 0.7192


## Experimental Results and Discussion

### Validation Results

The models were first compared using the validation dataset. Mean Absolute Error (MAE) was used as the primary model-selection criterion.

| Model | Validation MAE | Validation RMSE | Validation R² |
|---|---:|---:|---:|
| Stacking | **10.6714** | **27.3217** | **0.6400** |
| Equal-Weight Ensemble | 10.6773 | 27.8385 | 0.6263 |
| Hybrid RF + XGBoost | 10.9111 | 28.5140 | 0.6079 |
| XGBoost | 10.9345 | 27.4908 | 0.6355 |
| CatBoost | 11.0137 | 28.3088 | 0.6135 |
| Random Forest | 11.2937 | 30.7124 | 0.5451 |
| Baseline | 22.6346 | 46.9156 | -0.0615 |

*MAE and RMSE values are expressed in millions of Sri Lankan Rupees (LKR).*

**Interpretation:**

The Stacking Ensemble achieved the lowest validation MAE of 10.6714 million LKR, followed closely by the Equal-Weight Ensemble with an MAE of 10.6773 million LKR.

Among the individual machine learning models, XGBoost performed best, followed by CatBoost and Random Forest.

The baseline model produced a substantially higher MAE, demonstrating that the trained models learned useful relationships between property characteristics and advertised prices.

Stacking was selected for final evaluation because it achieved the lowest validation MAE. However, its improvement over Equal-Weight Averaging was only 0.0059 million LKR, indicating that the practical advantage was small.

### Final Test Results

After model selection, the trained models were evaluated using the previously unseen test dataset.

| Model | Test MAE | Test RMSE | Test R² |
|---|---:|---:|---:|
| Equal-Weight Ensemble | **9.9123** | 23.2264 | 0.7117 |
| Stacking | 9.9309 | **22.9200** | **0.7192** |
| Hybrid RF + XGBoost | 10.1125 | 23.5192 | 0.7044 |
| XGBoost | 10.2573 | 23.1249 | 0.7142 |
| CatBoost | 10.3163 | 24.5849 | 0.6770 |
| Random Forest | 10.3648 | 25.0012 | 0.6659 |
| Baseline | 21.7358 | 44.4029 | -0.0538 |

*MAE and RMSE values are expressed in millions of LKR.*

**Interpretation:**

The Equal-Weight Ensemble achieved the lowest test MAE of 9.9123 million LKR, slightly outperforming Stacking, which achieved 9.9309 million LKR.

However, Stacking achieved the lowest test RMSE of 22.9200 million LKR and the highest test R² of 0.7192.

These results show that Stacking performed particularly well in terms of squared-error-based evaluation, while simple averaging provided a competitive alternative with slightly better average absolute error.

The selected Stacking model was not changed after inspecting the test results because it had already been chosen using validation performance. This avoids selecting a final model based on the test dataset.

### Bagging vs Boosting

Random Forest achieved a test MAE of 10.3648 million LKR, while XGBoost achieved 10.2573 million LKR and CatBoost achieved 10.3163 million LKR.

Both Boosting models slightly outperformed Random Forest on test MAE.

This suggests that the sequential error-correction mechanisms used in Boosting were effective for this dataset. However, these findings apply to the specific models, configurations, and dataset used in this experiment; they do not establish that Boosting is always superior to Bagging.

### Stacking vs Hybridization

The Stacking Ensemble combined the three base learners using a trained Ridge Regression meta-learner.

The learned Ridge coefficients were approximately:

| Base Model | Ridge Coefficient |
|---|---:|
| Random Forest | 0.1038 |
| XGBoost | 0.5623 |
| CatBoost | 0.3642 |

XGBoost received the largest positive coefficient, suggesting that its predictions made a substantial contribution to the learned combination.

However, these coefficients are not normalized percentages. The meta-learner also includes an intercept.

Although Stacking achieved the best validation MAE, the Equal-Weight Ensemble produced very similar performance without requiring an additional trained model.

This indicates that simpler ensemble combinations may offer a useful balance between predictive performance and implementation complexity.

### Overall Findings

The experimental comparison produced the following findings:

1. All trained machine learning models substantially outperformed the median-prediction baseline.
2. XGBoost achieved the lowest test MAE among the individual trained models.
3. Combining multiple models improved the results compared with the individual models under several evaluation metrics.
4. Stacking achieved the best validation MAE and was selected as the final model.
5. Equal-Weight Averaging achieved the lowest test MAE, while Stacking achieved the best test RMSE and R².
6. The performance differences between the strongest ensembles were relatively small, so greater model complexity did not consistently produce a large improvement.

### Limitations

Several limitations should be considered when interpreting these results:

- The dataset contains historical advertised asking prices rather than verified completed property-sale prices.
- The model uses only five property characteristics and does not account for factors such as building condition, neighborhood accessibility, or market changes over time.
- Extreme-price filtering may remove some valid luxury or unusual properties.
- Duplicate-advertisement detection may not identify every repeated physical property.
- Model hyperparameters were not subjected to an extensive optimization study.
- Predictions may be unreliable for unusual properties or locations with limited training data.

Consequently, the developed system should be treated as an educational machine learning application rather than a professional property valuation service.

## Conclusion

This project successfully implemented and compared Bagging, Boosting, Stacking, and Hybrid Ensemble techniques for Sri Lankan house-price prediction.

Random Forest, XGBoost, and CatBoost were used as base learners, while Ridge Regression was employed to combine out-of-fold predictions in the Stacking Ensemble.

The Stacking Ensemble was selected using validation MAE and achieved a test R² of 0.7192 with a test RMSE of 22.9200 million LKR.

The findings also demonstrated that Equal-Weight Averaging can perform competitively against a trained stacking approach.

An interactive Streamlit dashboard was developed to visualize the dataset, compare model performance, explain the ensemble techniques, and generate predictions from the trained models.

In [ ]:

# SAVE MODELS AND DASHBOARD DATA
import joblib
import json
import zipfile
import pandas as pd

from pathlib import Path
from importlib.metadata import version
from google.colab import files

# 1. Create output folders
output_dir = Path("/content/ensemble_dashboard")
models_dir = output_dir / "models"
data_dir = output_dir / "data"

models_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

print("Saving trained models...")

# 2. Save individual trained models
joblib.dump(
    baseline_model,
    models_dir / "baseline.joblib"
)

joblib.dump(
    rf_model,
    models_dir / "random_forest.joblib"
)

joblib.dump(
    xgb_model,
    models_dir / "xgboost.joblib"
)

joblib.dump(
    cat_model,
    models_dir / "catboost.joblib"
)

# 3. Save stacking meta-learner
joblib.dump(
    meta_model,
    models_dir / "stacking_meta.joblib"
)

# 4. Save validation and test metrics
complete_comparison.to_csv(
    data_dir / "validation_results.csv",
    index=False
)

test_comparison.to_csv(
    data_dir / "test_results.csv",
    index=False
)

# 5. Save cleaned features for dashboard plots
dashboard_data = housing_unique[
    feature_columns + ["price_million"]
].copy()

dashboard_data.to_csv(
    data_dir / "housing_clean.csv",
    index=False
)

# 6. Save metadata
metadata = {
    "project": "Sri Lankan House Price Prediction",
    "selected_model": "Stacking",
    "base_model_order": list(base_models.keys()),
    "features": feature_columns,
    "target": "price_million",
    "price_unit": "Millions of LKR",
    "data_description": (
        "Historical Sri Lankan advertised house prices"
    ),
    "limitations": (
        "Predictions estimate historical advertised "
        "asking prices, not current transaction values."
    )
}

with open(
    output_dir / "metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=4)

# 7. Record package versions
packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "xgboost",
    "catboost",
    "joblib"
]

requirements = [
    f"{package}=={version(package)}"
    for package in packages
]

requirements += [
    "streamlit",
    "plotly"
]

(output_dir / "requirements.txt").write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

print("All files saved successfully!")

# 8. Create ZIP archive
zip_path = Path("/content/ensemble_dashboard.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for file in output_dir.rglob("*"):
        if file.is_file():
            zip_file.write(
                file,
                arcname=file.relative_to(output_dir)
            )

print("\nZIP created:", zip_path.name)

print("\nFiles included:")
for file in output_dir.rglob("*"):
    if file.is_file():
        print("-", file.relative_to(output_dir))

# 9. Download to your laptop
files.download(str(zip_path))


Saving trained models...
All files saved successfully!

ZIP created: ensemble_dashboard.zip

Files included:
- metadata.json
- requirements.txt
- models/xgboost.joblib
- models/catboost.joblib
- models/random_forest.joblib
- models/baseline.joblib
- models/stacking_meta.joblib
- data/test_results.csv
- data/validation_results.csv
- data/housing_clean.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# Export XGBoost in a portable format

import joblib
import pandas as pd
import zipfile

from pathlib import Path
from google.colab import files

# Create output directory
export_dir = Path("/content/xgb_portable")
export_dir.mkdir(exist_ok=True)

# Save the trained XGBoost model in native JSON format
xgb_model.named_steps["model"].save_model(
    str(export_dir / "xgboost_native.json")
)

# Save the fitted preprocessing pipeline separately
joblib.dump(
    xgb_model.named_steps["preprocessor"],
    export_dir / "xgboost_preprocessor.joblib"
)

# Save a few examples for verification
sample_inputs = X_val.head(5).copy()

sample_inputs.to_csv(
    export_dir / "xgb_check_inputs.csv",
    index=False
)

pd.DataFrame({
    "prediction": xgb_model.predict(sample_inputs)
}).to_csv(
    export_dir / "xgb_check_expected.csv",
    index=False
)

# Package all files into a ZIP
zip_path = "/content/xgb_portable.zip"

with zipfile.ZipFile(zip_path, "w") as archive:
    for file in export_dir.iterdir():
        archive.write(file, arcname=file.name)

print("XGBoost exported successfully!")

files.download(zip_path)


XGBoost exported successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>